# S6E9 | Charging and Income Feature Ablations

**Does a feature group help this model after the other features are already present?** Nine controlled HGB fits
compare all features, removal of charging-related columns, and removal of income/subsidy columns.

The verified Kaggle run found an AUC drop of about **0.079-0.081** without the economic group, but only **-0.000300 to
+0.000315** without the charging group. Small mixed-sign changes are not a stable win. Full-feature holdout AUC
was **0.941096 / 0.939690 / 0.939023** for stratified-17 / stratified-43 / ID-tail. The tables below are authoritative.

Reproducibility note: local and Kaggle inputs, sample IDs and partitions match, but library versions differ.
The maximum observed local/cloud AUC difference was **0.000272**, comparable to the charging-group effect itself.
This is another reason not to promote the tiny deletion effect. Exact package versions are saved in summary.json;
the public rerun must reproduce its reviewed Kaggle private run, not merely agree with the local trend.

## Fixed experiment

120,000 rows sampled once with seed 20260908. Two stratified holdouts (17 and 43), plus the highest-ID 20% as
a distribution stress test. Each split uses 96,000 training and 24,000 validation rows. Every variant reuses exactly
the same rows within its split, the same HGB settings and preprocessing fitted only on that split's training data.
ID and target are never model features. ID order is not assumed to be time.

Charging group: stations near home/work and home-charging availability. Economic group: annual income and subsidy
availability. These are **group-removal experiments**, not individual feature rankings or causal effects. The two
random splits overlap, so their range is not a confidence interval. No hyperparameter search or early stopping on
these validation labels. No test data, row predictions, fitted models or submission files are exported.


In [ ]:
import hashlib
import json
import os
import platform
import time
from pathlib import Path
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from threadpoolctl import threadpool_limits

OUTPUT = Path(os.environ.get('NB_OUTPUT', '/kaggle/working'))
OUTPUT.mkdir(parents=True, exist_ok=True)
roots = [Path(os.environ['EV_DATA_ROOT'])] if os.environ.get('EV_DATA_ROOT') else [
    Path('/kaggle/input/competitions/playground-series-s6e9'), Path('/kaggle/input/playground-series-s6e9')]
DATA = next(p for p in roots if (p / 'train.csv').is_file())
train_path = DATA / 'train.csv'
data = pd.read_csv(train_path)
TARGET = 'Will_Buy_EV'
assert data.id.is_unique and set(data[TARGET].unique()) == {'Yes', 'No'}
# Same sample budget as yesterday's CPU reference, with a new fixed sampling seed.
indices, _ = train_test_split(np.arange(len(data)), train_size=120000, random_state=20260908, stratify=data[TARGET])
frame = data.iloc[indices].sort_values('id').reset_index(drop=True)
del data
y = frame[TARGET].map({'No': 0, 'Yes': 1}).to_numpy()
groups = {'charging': ['Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Home_Charging_Possible'],
          'economic': ['Annual_Income_USD', 'Subsidy_Available']}
features = [c for c in frame if c not in ['id', TARGET]]
assert all(c in features for cols in groups.values() for c in cols)
variants = {'all_features': features, 'without_charging': [c for c in features if c not in groups['charging']],
            'without_economic': [c for c in features if c not in groups['economic']]}

## Fold-local preprocessing and fixed HGB

In [ ]:
def make_model(x):
    numeric = x.select_dtypes(include='number').columns.tolist()
    categorical = [c for c in x if c not in numeric]
    preprocess = ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), numeric),
        ('categorical', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
            ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical)])
    return Pipeline([('preprocess', preprocess), ('model', HistGradientBoostingClassifier(
        max_iter=120, learning_rate=.08, max_leaf_nodes=31, min_samples_leaf=40,
        l2_regularization=1., early_stopping=False, random_state=17))])


def fingerprint(idx):
    return hashlib.sha256(np.sort(frame.iloc[idx].id.to_numpy()).astype('<i8').tobytes()).hexdigest()

## Run paired feature-group removals

In [ ]:
rows, partitions = [], []
for split in ['stratified_17', 'stratified_43', 'id_tail']:
    if split == 'id_tail':
        tr, va = np.arange(96000), np.arange(96000, 120000)
    else:
        tr, va = train_test_split(np.arange(len(frame)), test_size=.2,
            random_state=int(split.split('_')[1]), stratify=y)
    assert not np.intersect1d(tr, va).size
    assert len(tr) == 96000 and len(va) == 24000
    partitions.append({'split': split, 'training_sha256': fingerprint(tr), 'validation_sha256': fingerprint(va)})
    for variant, cols in variants.items():
        assert 'id' not in cols and TARGET not in cols
        model = make_model(frame.iloc[tr][cols])
        started = time.perf_counter()
        with threadpool_limits(limits=2):
            model.fit(frame.iloc[tr][cols], y[tr])
            prediction = model.predict_proba(frame.iloc[va][cols])[:, 1]
        assert np.isfinite(prediction).all() and ((prediction >= 0) & (prediction <= 1)).all()
        row = {'split': split, 'variant': variant, 'features': len(cols),
               'auc': float(roc_auc_score(y[va], prediction)), 'log_loss': float(log_loss(y[va], prediction)),
               'fit_and_predict_seconds': time.perf_counter() - started}
        rows.append(row)
        print(split, variant, 'AUC', round(row['auc'], 6), flush=True)
results = pd.DataFrame(rows)
reference = results[results.variant == 'all_features'].set_index('split').auc
results['delta_auc_vs_all'] = results.auc - results.split.map(reference)
assert len(results) == 9
assert results.loc[results.variant == 'all_features', 'delta_auc_vs_all'].eq(0).all()

## Compare effects across the same splits

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
results.pivot(index='split', columns='variant', values='auc').plot.bar(
    ax=axes[0], rot=0, color=['#34699a', '#bb6c38', '#3a8464'])
axes[0].set(ylabel='Validation ROC-AUC', xlabel='', title='Fixed HGB model, nine fits')
axes[0].set_ylim(max(0, results.auc.min()-.01), min(1, results.auc.max()+.01))
results[results.variant != 'all_features'].pivot(index='split', columns='variant', values='delta_auc_vs_all').plot.bar(
    ax=axes[1], rot=0, color=['#bb6c38', '#3a8464'])
axes[1].axhline(0, color='black', linewidth=.8)
axes[1].set(ylabel='AUC change after removal', xlabel='', title='Paired feature-group ablations')
for ax in axes:
    ax.tick_params(axis='x', labelsize=8)
    ax.legend(fontsize=8)
fig.savefig(OUTPUT / 'feature_ablation.png', dpi=150)
plt.show()
results.to_csv(OUTPUT / 'scores.csv', index=False)
summary = {'observed_utc': datetime.now(timezone.utc).isoformat(), 'competition': 'playground-series-s6e9',
    'score_type': 'training_holdout_auc_not_leaderboard', 'train_sha256': hashlib.sha256(train_path.read_bytes()).hexdigest(),
    'sample_id_sha256': hashlib.sha256(frame.id.to_numpy().astype('<i8').tobytes()).hexdigest(),
    'sample_rows': len(frame), 'sampling_seed': 20260908, 'groups': groups, 'partitions': partitions,
    'packages': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'sklearn': sklearn.__version__},
    'scores': rows, 'test_or_evaluation_read': False, 'submission_created': False, 'leaderboard_score': None}
(OUTPUT / 'summary.json').write_text(json.dumps(summary, indent=2, allow_nan=False))
print(results[['split', 'variant', 'auc', 'delta_auc_vs_all']].to_string(index=False))
print('Run checks: PASS')

## Interpretation

Dropping the economic group removes information the fixed model relies on strongly. It does not establish that
income causes purchases, nor isolate income from subsidy. Redundant features can make group removal understate
the usefulness of any single column. The small charging-group deltas change sign across splits; retain that
negative/ambiguous result rather than recommending feature deletion from one holdout.

Validation AUC here is not a leaderboard score, and Playground does not award Competition medals. A production
change needs an independently reserved holdout or OOF experiment; do not select a final model from these nine fits.
The full AUC panel uses a restricted y-axis for comparison; use the paired-difference panel for effect size.

## Sources

- [Official S6E9 data](https://www.kaggle.com/competitions/playground-series-s6e9/data) and
  [rules](https://www.kaggle.com/competitions/playground-series-s6e9/rules), observed 2026-09-08.
- [Previous CPU model comparison](https://www.kaggle.com/code/muelsyse111/s6e9-cpu-baselines-and-paired-auc).
  Same fixed HGB configuration; a different, explicitly seeded sample. Scores are not paired with yesterday's run.
- [Previous validation stress tests](https://www.kaggle.com/code/muelsyse111/s6e9-validation-stability-and-id-order-tests).
  This notebook changes feature availability while holding the model fixed, rather than comparing split protocols alone.
- scikit-learn supplies preprocessing, HGB, splits and metrics. No other author's model code is copied.

Prepared with AI assistance. Inputs were listed as created 2026-08-12, not new today. Only aggregate outputs are saved.
